# Module 05 — Data Locality, Shuffle & SPJ

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

spark = begin_module('module-05-data-locality')
ui_links()

## Why shuffle appears

In [ ]:
from pyspark.sql import functions as F, Window
trips = spark.read.parquet(*config.yellow_taxi_paths())
agg = trips.groupBy('PULocationID', 'VendorID').count()
agg.explain('formatted')  # Exchange
w = Window.partitionBy('VendorID').orderBy(F.desc('total_amount'))
win = trips.withColumn('rk', F.row_number().over(w))
win.limit(1).explain('formatted')

## Query shape — keep join keys in aggregation

In [ ]:
users, events = user_post_activity(spark)
joined = events.join(users, 'user_id')
bad = joined.groupBy('post_id').count()  # may reshuffle post_id alone
good = joined.groupBy('post_id', 'user_id').count()  # aligned with join key
bad.explain('formatted')
good.explain('formatted')

## Join hints

In [ ]:
zones = spark.read.option('header', True).csv(config.zones_path())
z = zones.withColumnRenamed('LocationID', 'PULocationID')
from pyspark.sql.functions import broadcast
hinted = trips.join(broadcast(z), 'PULocationID', 'inner')
hinted.limit(1).explain('formatted')

## Storage Partitioned Join — setup

In [ ]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'true')
spark.conf.set('spark.sql.sources.v2.bucketing.pushPartValues.enabled', 'true')
spark.conf.set('spark.sql.iceberg.planning.preserve-data-grouping', 'true')
spark.conf.set('spark.sql.requireAllClusterKeysForCoPartition', 'false')
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')

base = trips.select('VendorID', 'PULocationID', 'total_amount')
drop_iceberg_table(spark, 'local.default.spj_a')
drop_iceberg_table(spark, 'local.default.spj_b')
base.writeTo('local.default.spj_a').using('iceberg').partitionedBy('PULocationID').create()
base.writeTo('local.default.spj_b').using('iceberg').partitionedBy('PULocationID').create()

### Vanilla SPJ

In [ ]:
spj = spark.sql('''SELECT a.VendorID, a.total_amount, b.total_amount AS amt_b
    FROM local.default.spj_a a JOIN local.default.spj_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
spj.explain('formatted')  # no Exchange before join = success

### SPJ off — Exchange returns

In [ ]:
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'false')
no_spj = spark.sql('''SELECT a.VendorID FROM local.default.spj_a a JOIN local.default.spj_b b
    ON a.PULocationID = b.PULocationID AND a.VendorID = b.VendorID''')
no_spj.explain('formatted')
spark.conf.set('spark.sql.sources.v2.bucketing.enabled', 'true')

### One-sided SPJ (only one table partition-aligned)

Join Iceberg partitioned table to ad-hoc Parquet — expect shuffle on the unclustered side.

In [ ]:
side_path = str(config.OUTPUT_BASE / '05-side.parquet')
base.write.mode('overwrite').parquet(side_path)
side = spark.read.parquet(side_path)
mixed = spark.table('local.default.spj_a').join(side, ['PULocationID', 'VendorID'])
mixed.limit(1).explain('formatted')

## Takeaways

1. Exchange marks shuffle — agg/join/window often need it.
2. Align agg keys with prior join keys when possible.
3. SPJ when both inputs share partition layout; compare with bucketing disabled.